# APOE-independent credible sets and neuropathology association analysis
## Aims
 - This notebook tests whether APOE-independent credible sets (CS) are associated with neuropathological outcomes in ROSMAP.
 - Original script by Alexandre (`APOE_cs_neuropatho_fromAlexandre.R`).

## Conclusions
We can validate 3 of the 19 independent sets explaining one neuropathology after adjusted for APOE4 and 2 dose (set 27, 31, and 57), and 4/19 sets when we only adjust for APOE4. These results are at FDR < 0.05, using BH correction with n = 19 tests. \
Breakdown of the neuropathology associated sets:
 - **set_27** is the APOE Mic eQTL that have previously been found associated with cerebral amyloid angiopathy in Fujita et al., and we are able to recapitulate this association (also w/ TDP-43 pathology)
 - **set_26**: eQTL of APOC1 (in several brain regions and STARNET Mac), explaining age at AD onset
 - **set_31**: eQTL of BLOC1S3 (in several brain regions), explaining age at AD onset
 - **set_57**: eQTL of ARHGAP35 (in Ast.10 specifically), explaining Tau tangle level (mediation: variants > ARHGAP35 expression > Tau tangles)

**Questions:**
1. Does APOE-independent mean APOE4/APOE2-independent? Both
2. Are selected APOE-independent CSs significantly associated with AD in GWAS? Not necessarily
3. What is the definition of the outcome in AD GWAS? Standard diagnostic criteria
4. What is 'complete' dataset? `/mnt/lustre/lab/gwang/data/ftp_lisanwanglab_sync/ftp_fgc_xqtl/ref-data/ROSMAP_covariates/ROSMAP_xqtl_complete_samples_covariates_sex_death_pmi_study.tsv`
5. In neuropathology dataset, projid vs iid? Should be the same
6. Rationale for picking neuropathology outcomes?
7. Why including APOE4/2 dose as outcomes? To check how APOE4 and 2 correlate with the tested variants in this dataset

## 1. Setup and prepare the input

In [2]:
source('/mnt/lustre/home/yl4437/xqtl_flagship/APOE/r_utils.R')
library(tibble)
library(dplyr)

setwd('/mnt/lustre/home/yl4437/xqtl_flagship/APOE')
out <- '/mnt/lustre/home/yl4437/xqtl_flagship/APOE'

1 threads available for data.table

qs 0.27.3. Announcement: https://github.com/qsbase/qs/issues/103


Attaching package: ‘dplyr’


The following objects are masked from ‘package:data.table’:

    between, first, last


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




### Prepare genotype data

In [3]:
# Load the independent credible sets and format them into a data.table with variant IDs, GWAS source, and molecular outcome information
res_indep_cs <- readRDS('/mnt/lustre/home/yl4437/xqtl_flagship/xqtl-paper/main_text/5_AD_xQTL_genes_cis_trans/figure_data/xqtl_only_APOE_all_cohorts_independent_sets_after_imputation_no_sQTL_only.rds')
str(res_indep_cs)
length(res_indep_cs$variant_id)

List of 3
 $ variant_id:List of 19
  ..$ ind_set_after_between_purity_145: chr [1:18] "chr19:44883377:C:T" "chr19:44885307:CTTTTTT:C" "chr19:44885313:T:*" "chr19:44885317:T:C" ...
  ..$ ind_set_after_between_purity_27 : chr [1:90] "chr19:44980181:A:C" "chr19:44986934:A:G" "chr19:44987312:A:G" "chr19:44989803:T:C" ...
  ..$ ind_set_after_between_purity_57 : chr [1:2] "chr19:44827033:TGATAGATAGATAGATA:TGATA" "chr19:44840322:G:A"
  ..$ ind_set_after_between_purity_108: chr "chr19:45169220:G:A"
  ..$ ind_set_after_between_purity_192: chr [1:8] "chr19:44822481:T:G" "chr19:44824202:T:C" "chr19:44834480:C:T" "chr19:44834606:T:C" ...
  ..$ ind_set_after_between_purity_26 : chr [1:5] "chr19:44918487:G:T" "chr19:44918715:AG:A" "chr19:44920379:G:A" "chr19:44925202:C:T" ...
  ..$ ind_set_after_between_purity_135: chr "chr19:45547502:T:C"
  ..$ ind_set_after_between_purity_69 : chr [1:2] "chr19:43090637:G:T" "chr19:42994877:C:G"
  ..$ ind_set_after_between_purity_197: chr [1:28] "chr19:45081719:C:T

[1] 19

In [4]:
indep_cs <- rbindlist(lapply(names(res_indep_cs$variant_id), function(n)data.table(cs_name = n, variant_id = res_indep_cs$variant_id[[n]],
                                                                                   gwas_source = res_indep_cs$ad_gwas[[which(names(res_indep_cs$variant_id)==n)]],
                                                                                   molecular_outcome = res_indep_cs$outcome[[n]])))
indep_cs[,cs_id:=paste0('set_',str_extract(cs_name,'[0-9]+$'))]
indep_cs[,chr:=seqid(variant_id)]
indep_cs[,pos:=pos(variant_id)]
indep_cs[,variant_id2:=str_replace_all(variant_id,':','_')|>str_replace('_',':')]

In [4]:
head(indep_cs); nrow(indep_cs)

cs_name,variant_id,gwas_source,molecular_outcome,cs_id,chr,pos,variant_id2
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
ind_set_after_between_purity_145,chr19:44883377:C:T,AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021; AD_Bellenguez_EADB,DLPFC_Klein_gpQTL_unadjusted_gp_0860_O96005_ENSG00000104853,set_145,chr19,44883377,chr19:44883377_C_T
ind_set_after_between_purity_145,chr19:44885307:CTTTTTT:C,AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021; AD_Bellenguez_EADB,DLPFC_Klein_gpQTL_unadjusted_gp_0860_O96005_ENSG00000104853,set_145,chr19,44885307,chr19:44885307_CTTTTTT_C
ind_set_after_between_purity_145,chr19:44885313:T:*,AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021; AD_Bellenguez_EADB,DLPFC_Klein_gpQTL_unadjusted_gp_0860_O96005_ENSG00000104853,set_145,chr19,44885313,chr19:44885313_T_*
ind_set_after_between_purity_145,chr19:44885317:T:C,AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021; AD_Bellenguez_EADB,DLPFC_Klein_gpQTL_unadjusted_gp_0860_O96005_ENSG00000104853,set_145,chr19,44885317,chr19:44885317_T_C
ind_set_after_between_purity_145,chr19:44893716:G:A,AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021; AD_Bellenguez_EADB,DLPFC_Klein_gpQTL_unadjusted_gp_0860_O96005_ENSG00000104853,set_145,chr19,44893716,chr19:44893716_G_A
ind_set_after_between_purity_145,chr19:44894050:C:T,AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021; AD_Bellenguez_EADB,DLPFC_Klein_gpQTL_unadjusted_gp_0860_O96005_ENSG00000104853,set_145,chr19,44894050,chr19:44894050_C_T


[1] 216

In [5]:
# Extract genotype for CS snps (PLINK to extract genotypes from the ROSMAP WGS data & recode to VCF format)
fwrite(indep_cs[order(chr,pos)][,.(variant_id2)], fp(out,'apoe4_indep_snps.tsv'), sep = '\t', col.names = FALSE)

plink_file <- '/mnt/lustre/lab/gwang/ftp_fgc_xqtl/ROSMAP/genotype/analysis_ready/geno_by_chrom/ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.19'

cmd = paste('plink --bfile', plink_file,
            '--extract', fp(out,'apoe4_indep_snps.tsv'),
            '--recode vcf',
            '--out', fp(out,'ROSMAP_geno_apoe4_indep_snps'))

system(cmd)

In [5]:
# Format the genotype (melt to long format with one row per donor-variant, compute allele dosage, and merge with the credible set information)
donors_apoecs <- fread(fp(out, 'ROSMAP_geno_apoe4_indep_snps.vcf'))
dim(donors_apoecs)

[1]  216 1162

In [6]:
donors_apoecs <- melt(donors_apoecs[,-c(6:9)],
                      id.vars = c('#CHROM','POS','ID','REF','ALT'),
                      variable.name = 'IID', value.name = 'geno')
donors_apoecs[,dose:=str_count(geno, '1')]

donors_apoecs[,variant_id2:=ID]
donors_apoecs <- merge(donors_apoecs, indep_cs, by = 'variant_id2')
head(donors_apoecs); dim(donors_apoecs)

variant_id2,#CHROM,POS,ID,REF,ALT,IID,geno,dose,cs_name,variant_id,gwas_source,molecular_outcome,cs_id,chr,pos
<chr>,<int>,<int>,<chr>,<chr>,<chr>,<fct>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
chr19:42994877_C_G,19,42994877,chr19:42994877_C_G,C,G,0_MAP15387421,0/0,0,ind_set_after_between_purity_69,chr19:42994877:C:G,AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021,BM_10_MSBB_eQTL_ENSG00000176472; monocyte_ROSMAP_eQTL_ENSG00000130755,set_69,chr19,42994877
chr19:42994877_C_G,19,42994877,chr19:42994877_C_G,C,G,0_MAP26637867,0/0,0,ind_set_after_between_purity_69,chr19:42994877:C:G,AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021,BM_10_MSBB_eQTL_ENSG00000176472; monocyte_ROSMAP_eQTL_ENSG00000130755,set_69,chr19,42994877
chr19:42994877_C_G,19,42994877,chr19:42994877_C_G,C,G,0_MAP29629849,0/1,1,ind_set_after_between_purity_69,chr19:42994877:C:G,AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021,BM_10_MSBB_eQTL_ENSG00000176472; monocyte_ROSMAP_eQTL_ENSG00000130755,set_69,chr19,42994877
chr19:42994877_C_G,19,42994877,chr19:42994877_C_G,C,G,0_MAP33332646,0/0,0,ind_set_after_between_purity_69,chr19:42994877:C:G,AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021,BM_10_MSBB_eQTL_ENSG00000176472; monocyte_ROSMAP_eQTL_ENSG00000130755,set_69,chr19,42994877
chr19:42994877_C_G,19,42994877,chr19:42994877_C_G,C,G,0_MAP34726040,0/0,0,ind_set_after_between_purity_69,chr19:42994877:C:G,AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021,BM_10_MSBB_eQTL_ENSG00000176472; monocyte_ROSMAP_eQTL_ENSG00000130755,set_69,chr19,42994877
chr19:42994877_C_G,19,42994877,chr19:42994877_C_G,C,G,0_MAP46246604,0/0,0,ind_set_after_between_purity_69,chr19:42994877:C:G,AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021,BM_10_MSBB_eQTL_ENSG00000176472; monocyte_ROSMAP_eQTL_ENSG00000130755,set_69,chr19,42994877


[1] 249048     16

### Prepare covariate data

In [7]:
# Add the clinical covariates (ROSMAP clinical data, biospecimen metadata for WGS sample-to-individual mapping, and covariates including sex, death, PMI, study)
clin <- fread('/mnt/lustre/lab/gwang/ftp_fgc_xqtl/ROSMAP/metadata/ROSMAP_clinical.csv')
bios <- fread('/mnt/lustre/lab/gwang/ftp_fgc_xqtl/ROSMAP/metadata/ROSMAP_biospecimen_metadata.csv') #for WGS samples ID - IID link
clinw <- merge(clin, bios[assay == 'wholeGenomeSeq'][,.(individualID, specimenID)], by = 'individualID')
head(clinw); nrow(clinw)

individualID,projid,Study,msex,educ,race,spanish,apoe_genotype,age_at_visit_max,age_first_ad_dx,age_death,cts_mmse30_first_ad_dx,cts_mmse30_lv,pmi,braaksc,ceradsc,cogdx,dcfdx_lv,specimenID
<chr>,<int>,<chr>,<int>,<int>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<int>,<int>,<int>,<chr>
R1012422,20925446,ROS,0,14,1,2,33,87.501711156741962,,87.55099247091033,NA,14,3.000000,4,2,4,4,SM-CTEMF
R1015854,45115248,MAP,0,16,1,2,33,86.811772758384663,,87.115674195756327,NA,26,4.633333,2,4,2,2,SM-CTDQP
R1017692,20787136,ROS,0,16,1,2,33,74.135523613963045,,75.019849418206704,NA,27,4.000000,4,1,2,4,SM-CJFP9
R1018391,50103381,MAP,1,16,1,2,34,89.639972621492134,,90+,NA,29,5.416667,4,2,2,2,SM-CJIXL
R1020037,20154287,ROS,0,20,1,2,23,89.073237508555778,87.00342231348391,90+,28,26,2.750000,3,4,2,2,SM-CJEJ2
R1022980,30819298,MAP,0,10,1,2,33,89.111567419575636,,89.900068446269685,NA,8,8.083333,3,3,4,4,SM-CTEMK


[1] 1196

In [8]:
# Original: /projectnb/tcwlab/ShareSpace/ROSMAP_Neuropatho/ROSMAP_xqtl_complete_samples_covariates_sex_death_pmi_study.csv
complete <- fread('/mnt/lustre/lab/gwang/data/ftp_lisanwanglab_sync/ftp_fgc_xqtl/ref-data/ROSMAP_covariates/ROSMAP_xqtl_complete_samples_covariates_sex_death_pmi_study.tsv')
complete <- complete %>% t() %>%  as.data.frame() %>% rownames_to_column()
colnames(complete) <- complete[1, ]
complete <- complete[-1, ]
colnames(complete)[1] <- 'specimenID'
complete <- complete %>% mutate(across(-specimenID, as.numeric))
complete <- na.omit(complete)
head(complete); dim(complete)

,specimenID,msex,age_death,pmi,ROS_study
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
2,SM-CJFK8,0,100.87340,7.500000,0
3,SM-CTDUU,0,80.65708,5.500000,0
4,SM-CTDSC,0,83.69062,3.933333,0
5,SM-CJIYB,1,92.61875,5.083333,0
6,SM-CTEMS,0,90.38467,3.783333,0
7,SM-CJFLR,0,94.31348,5.000000,0


[1] 1158    5

In [9]:
clinw <- merge(clinw[,-c('age_death','pmi','msex')], complete, by = 'specimenID')
donors_apoecs[,specimenID:=str_remove(IID,'0_')]
donors_apoecs <- merge(donors_apoecs, clinw, by = 'specimenID')
unique(donors_apoecs$individualID)|>length() # Different sample sizes: 1116 vs 1113

[1] 1113

### Prepare neuropathology data

In [10]:
# Neuropathology covs: amyloid, tangles, plaques, NFT, TDP-43, CAA, cognitive decline slope
# Test only a subset of the covariates
neuropath <- fread('/mnt/lustre/home/yl4437/resource/RM_pheno_dataset_707_basic_02-10-2022.n3711.clean.plink.n2409.28FEB2022.txt')
dim(neuropath)
neuropathf <- neuropath[,.SD,.SDcols = c('projid','amyloid_sqrt','tangles_sqrt','cogng_path_slope','plaq_d','plaq_n_sqrt','nft_sqrt','tdp_cs_6reg','tdp_dn_6reg','tdp_st4','caa_4gp','caa_neo4')]

# Merge phenotype and genotypes
donors_apoecs <- merge(donors_apoecs, neuropathf, all.x = TRUE, by = 'projid')

[1] 2409  266

In [11]:
head(donors_apoecs); colnames(donors_apoecs); dim(donors_apoecs)

projid,specimenID,variant_id2,#CHROM,POS,ID,REF,ALT,IID,geno,⋯,tangles_sqrt,cogng_path_slope,plaq_d,plaq_n_sqrt,nft_sqrt,tdp_cs_6reg,tdp_dn_6reg,tdp_st4,caa_4gp,caa_neo4
<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<fct>,<chr>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<int>,<dbl>
246264,SM-CJFK8,chr19:42994877_C_G,19,42994877,chr19:42994877_C_G,C,G,0_SM-CJFK8,0/0,⋯,2.218849,0.07273753,4.195389,0.7917066,0.594267,0.8333333,0.6666667,3,2,2.25
246264,SM-CJFK8,chr19:43090637_G_T,19,43090637,chr19:43090637_G_T,G,T,0_SM-CJFK8,0/0,⋯,2.218849,0.07273753,4.195389,0.7917066,0.594267,0.8333333,0.6666667,3,2,2.25
246264,SM-CJFK8,chr19:43337238_G_A,19,43337238,chr19:43337238_G_A,G,A,0_SM-CJFK8,0/0,⋯,2.218849,0.07273753,4.195389,0.7917066,0.594267,0.8333333,0.6666667,3,2,2.25
246264,SM-CJFK8,chr19:43522968_T_C,19,43522968,chr19:43522968_T_C,T,C,0_SM-CJFK8,0/0,⋯,2.218849,0.07273753,4.195389,0.7917066,0.594267,0.8333333,0.6666667,3,2,2.25
246264,SM-CJFK8,chr19:43632995_C_T,19,43632995,chr19:43632995_C_T,C,T,0_SM-CJFK8,0/0,⋯,2.218849,0.07273753,4.195389,0.7917066,0.594267,0.8333333,0.6666667,3,2,2.25
246264,SM-CJFK8,chr19:43656997_G_A,19,43656997,chr19:43656997_G_A,G,A,0_SM-CJFK8,0/0,⋯,2.218849,0.07273753,4.195389,0.7917066,0.594267,0.8333333,0.6666667,3,2,2.25


[1] "projid"                 "specimenID"             "variant_id2"           
 [4] "#CHROM"                 "POS"                    "ID"                    
 [7] "REF"                    "ALT"                    "IID"                   
[10] "geno"                   "dose"                   "cs_name"               
[13] "variant_id"             "gwas_source"            "molecular_outcome"     
[16] "cs_id"                  "chr"                    "pos"                   
[19] "individualID"           "Study"                  "educ"                  
[22] "race"                   "spanish"                "apoe_genotype"         
[25] "age_at_visit_max"       "age_first_ad_dx"        "cts_mmse30_first_ad_dx"
[28] "cts_mmse30_lv"          "braaksc"                "ceradsc"               
[31] "cogdx"                  "dcfdx_lv"               "msex"                  
[34] "age_death"              "pmi"                    "ROS_study"             
[37] "amyloid_sqrt"           "tangles_sqrt"           "cogng_path_slope"      
[40] "plaq_d"                 "plaq_n_sqrt"            "nft_sqrt"              
[43] "tdp_cs_6reg"            "tdp_dn_6reg"            "tdp_st4"               
[46] "caa_4gp"                "caa_neo4"

[1] 240408     47

### Reformat the input
 - Compute APOE4/APOE2 dosage
 - Binarize and categorize cognitive diagnosis
 - Derive additional clinical variables
 - Compute correlation of each CS variant dosage with APOE2/APOE4 dosage
 - Save the combined dataset

In [12]:
# Neuropathology analysis correcting for APOE4/2 genotype
donors_apoecs[,apoe4_dose:=str_count(apoe_genotype,'4')]
donors_apoecs[,apoe2_dose:=str_count(apoe_genotype,'2')]

# Prep/reformat the phenotypes
donors_apoecs[,cogdx_binary:=ifelse(cogdx%in%c(4,5),'AD',ifelse(cogdx%in%1:3,'No AD',NA))] # Binarize the cognitive status

donors_apoecs[,cogdx_merge:=ifelse(cogdx%in%c(4,5),'AD',ifelse(cogdx%in%2:3,'MCI',ifelse(cogdx==1,'NL',NA)))] # Three level cognitive status
donors_apoecs[,cogdx_merge:=factor(cogdx_merge,levels = c('NL','MCI','AD'))]
donors_apoecs[cogdx!=6,cogdx_mergenum:=ifelse(cogdx_merge=='MCI',2,ifelse(cogdx_merge=='AD',3,1))]

donors_apoecs[,age_first_ad_dx_num:=ifelse(age_first_ad_dx=='90+',91,as.numeric(age_first_ad_dx))]
donors_apoecs[,other_dementia:=cogdx==6]
donors_apoecs[,other_CI:=cogdx%in%c(3,5)]
donors_apoecs[cogdx%in%c(1,4,5),ADvsNL:=cogdx%in%c(4,5)]
donors_apoecs[cogdx%in%c(1,2,3),MCIvsNL:=cogdx%in%c(2,3)]

donors_apoecs[,ceradsc_pos:=-ceradsc]
donors_apoecs[,braaksc06:=ifelse(braaksc==0,1,ifelse(braaksc==6,5,braaksc))]

# donors_apoecs[,age_first_ad_dx_num:=ifelse(age_first_ad_dx=='90+',91,as.numeric(age_first_ad_dx))] # Duplicated rows
donors_apoecs[(cogdx_binary=='AD'),age_first_clinad_dx:=age_first_ad_dx_num]
donors_apoecs[(cogdx_binary=='AD'),cts_mmse30_lv_clinad:=cts_mmse30_lv]

donors_apoecs[,age_at_visit_num:=ifelse(age_at_visit_max=='90+',91,as.numeric(age_at_visit_max))]
donors_apoecs[dcfdx_lv!=6,dcfdx_lv_no6:=dcfdx_lv]

donors_apoecs[,apoe2_r:=cor(dose,apoe2_dose,use = 'pairwise.complete.obs'),by='ID']
donors_apoecs[,apoe4_r:=cor(dose,apoe4_dose,use = 'pairwise.complete.obs'),by='ID']

head(donors_apoecs)
fwrite(donors_apoecs,fp(out,'ROSMAP_geno_apoecs_adpathology.csv.gz'))

Warning message in ifelse(age_first_ad_dx == "90+", 91, as.numeric(age_first_ad_dx)):
“NAs introduced by coercion”
Warning message in ifelse(age_at_visit_max == "90+", 91, as.numeric(age_at_visit_max)):
“NAs introduced by coercion”


projid,specimenID,variant_id2,#CHROM,POS,ID,REF,ALT,IID,geno,⋯,ADvsNL,MCIvsNL,ceradsc_pos,braaksc06,age_first_clinad_dx,cts_mmse30_lv_clinad,age_at_visit_num,dcfdx_lv_no6,apoe2_r,apoe4_r
<int>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<fct>,<chr>,⋯,<lgl>,<lgl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>
246264,SM-CJFK8,chr19:42994877_C_G,19,42994877,chr19:42994877_C_G,C,G,0_SM-CJFK8,0/0,⋯,FALSE,FALSE,-2,3,NA,NA,91,1,0.019146904,0.04481688
246264,SM-CJFK8,chr19:43090637_G_T,19,43090637,chr19:43090637_G_T,G,T,0_SM-CJFK8,0/0,⋯,FALSE,FALSE,-2,3,NA,NA,91,1,0.025140354,0.03118963
246264,SM-CJFK8,chr19:43337238_G_A,19,43337238,chr19:43337238_G_A,G,A,0_SM-CJFK8,0/0,⋯,FALSE,FALSE,-2,3,NA,NA,91,1,0.030118083,0.02325655
246264,SM-CJFK8,chr19:43522968_T_C,19,43522968,chr19:43522968_T_C,T,C,0_SM-CJFK8,0/0,⋯,FALSE,FALSE,-2,3,NA,NA,91,1,0.002532300,0.03126567
246264,SM-CJFK8,chr19:43632995_C_T,19,43632995,chr19:43632995_C_T,C,T,0_SM-CJFK8,0/0,⋯,FALSE,FALSE,-2,3,NA,NA,91,1,0.008980595,0.01017905
246264,SM-CJFK8,chr19:43656997_G_A,19,43656997,chr19:43656997_G_A,G,A,0_SM-CJFK8,0/0,⋯,FALSE,FALSE,-2,3,NA,NA,91,1,-0.013580187,0.04814827


In [33]:
summary(donors_apoecs)

     projid          specimenID        variant_id2            #CHROM  
 Min.   :  246264   Length:240408      Length:240408      Min.   :19  
 1st Qu.:20153010   Class :character   Class :character   1st Qu.:19  
 Median :21172121   Mode  :character   Mode  :character   Median :19  
 Mean   :35645689                                         Mean   :19  
 3rd Qu.:50400385                                         3rd Qu.:19  
 Max.   :99746564                                         Max.   :19  
                                                                      
      POS                ID                REF                ALT           
 Min.   :42994877   Length:240408      Length:240408      Length:240408     
 1st Qu.:44869362   Class :character   Class :character   Class :character  
 Median :44945575   Mode  :character   Mode  :character   Mode  :character  
 Mean   :44901146                                                           
 3rd Qu.:44967230                              

## 2. Regression

### Define outcomes for regression
Specify which outcomes are tested with linear models vs logistic regression & which require specific covariate adjustments (e.g., age at last visit vs age at death)

In [13]:
# Outcomes to test with linear regression
covs_lin <- c('ceradsc_pos','braaksc06','dcfdx_lv_no6','cts_mmse30_lv','age_death','cogdx_mergenum','apoe4_dose','apoe2_dose',
              'cts_mmse30_lv_clinad','age_first_ad_dx_num','age_first_clinad_dx','amyloid_sqrt','tangles_sqrt','cogng_path_slope',
              'plaq_d','plaq_n_sqrt','nft_sqrt','tdp_cs_6reg','tdp_dn_6reg','tdp_st4','caa_4gp','caa_neo4')

# Outcomes to test with logistic regression
covs_logis <- c('MCIvsNL','ADvsNL')

# Outcomes that are measured before death
atlastvisit <- c('cts_mmse30_lv','cts_mmse30_lv_clinad') # Outcomes that need to be adjusted for age at last visit
beforedeath <- c('age_death','cts_mmse30_lv','cts_mmse30_lv_clinad','age_first_ad_dx_num','age_first_clinad_dx','cogng_path_slope')

### Regression 1: Adjusting for both APOE4 and APOE2
Run linear models (for continuous/ordinal outcomes) and logistic regression (for binary outcomes) for each CS variant against each neuropathological phenotype, adjusting for sex, education, age, and both APOE4 and APOE2 dosage.

In [14]:
apoecs_clin <- rbindlist(lapply(covs_lin, function(outc){
  message(outc)
  if(outc %in% atlastvisit){
    donors_apoecs[,summary(lm(unlist(.SD)~dose+msex+educ+age_at_visit_num+apoe4_dose+apoe2_dose))$coefficients|>data.table(keep.rownames = 'cov'),
                  by='variant_id2',.SDcols=outc][cov=='dose'][,outcome:=outc]
  }else if(outc %in% beforedeath){
    donors_apoecs[,summary(lm(unlist(.SD)~dose+msex+educ+apoe4_dose+apoe2_dose))$coefficients|>data.table(keep.rownames = 'cov'),
                  by='variant_id2',.SDcols=outc][cov=='dose'][,outcome:=outc]
  }else{
    donors_apoecs[,summary(lm(unlist(.SD)~dose+msex+educ+age_death+apoe4_dose+apoe2_dose))$coefficients|>data.table(keep.rownames = 'cov'),
                  by='variant_id2',.SDcols=outc][cov=='dose'][,outcome:=outc]
  }
}))

apoecs_clin[,zscore:=`t value`]
apoecs_clin[,pval:=`Pr(>|t|)`]

ceradsc_pos



braaksc06

dcfdx_lv_no6

cts_mmse30_lv

age_death

cogdx_mergenum

apoe4_dose

Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: sum

In [36]:
apoecs_clogit <- rbindlist(lapply(covs_logis, function(outc){
  message(outc)
  donors_apoecs[,summary(glm(unlist(.SD)~dose+msex+educ+age_death+apoe4_dose+apoe2_dose,family = binomial))$coefficients|>data.table(keep.rownames = 'cov'),
              by='variant_id2',.SDcols=outc][cov=='dose'][,outcome:=outc] 
}))

apoecs_clogit[,zscore:=`z value`]
apoecs_clogit[,pval:=`Pr(>|z|)`]

MCIvsNL

ADvsNL



In [37]:
apoecs_clin <- rbind(apoecs_clin, apoecs_clogit, fill = TRUE)
apoecs_clin <- merge(apoecs_clin, indep_cs, by = 'variant_id2')
apoecs_clin[,padj:=p.adjust(pval,n=length(unique(indep_cs$cs_name))),by=c('outcome','variant_id2')]
head(apoecs_clin[padj<0.05])

variant_id2,cov,Estimate,Std. Error,t value,Pr(>|t|),outcome,zscore,pval,z value,Pr(>|z|),cs_name,variant_id,gwas_source,molecular_outcome,cs_id,chr,pos,padj
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
chr19:43337238_G_A,dose,-1.168904e-17,2.012350e-18,-5.808650,8.239442e-09,apoe2_dose,-5.808650,8.239442e-09,NA,NA,ind_set_after_between_purity_103,chr19:43337238:G:A,AD_Wightman_ExcludingUKBand23andME_2021,BM_44_MSBB_eQTL_ENSG00000130755,set_103,chr19,43337238,1.565494e-07
chr19:44723473_T_C,dose,-3.134197e-17,6.054598e-18,-5.176556,2.686009e-07,apoe2_dose,-5.176556,2.686009e-07,NA,NA,ind_set_after_between_purity_267,chr19:44723473:T:C,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44723473,5.103417e-06
chr19:44724393_G_T,dose,-3.134197e-17,6.054598e-18,-5.176556,2.686009e-07,apoe2_dose,-5.176556,2.686009e-07,NA,NA,ind_set_after_between_purity_267,chr19:44724393:G:T,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44724393,5.103417e-06
chr19:44724456_A_G,dose,-3.134197e-17,6.054598e-18,-5.176556,2.686009e-07,apoe2_dose,-5.176556,2.686009e-07,NA,NA,ind_set_after_between_purity_267,chr19:44724456:A:G,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44724456,5.103417e-06
chr19:44724557_C_T,dose,-3.134197e-17,6.054598e-18,-5.176556,2.686009e-07,apoe2_dose,-5.176556,2.686009e-07,NA,NA,ind_set_after_between_purity_267,chr19:44724557:C:T,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44724557,5.103417e-06
chr19:44724638_C_A,dose,-3.134197e-17,6.054598e-18,-5.176556,2.686009e-07,apoe2_dose,-5.176556,2.686009e-07,NA,NA,ind_set_after_between_purity_267,chr19:44724638:C:A,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44724638,5.103417e-06


In [38]:
dim(apoecs_clin); colnames(apoecs_clin)

[1] 5184   19

[1] "variant_id2"       "cov"               "Estimate"         
 [4] "Std. Error"        "t value"           "Pr(>|t|)"         
 [7] "outcome"           "zscore"            "pval"             
[10] "z value"           "Pr(>|z|)"          "cs_name"          
[13] "variant_id"        "gwas_source"       "molecular_outcome"
[16] "cs_id"             "chr"               "pos"              
[19] "padj"

In [50]:
# Filter significant results (padj < 0.05) excluding APOE dosage outcomes & count the number of significant CS
# apoecs_clin[!outcome %in% c('apoe4_dose','apoe2_dose')][padj<0.05]
apoecs_clin[padj<0.05][!outcome %in% c('apoe4_dose','apoe2_dose')]$cs_id|>unique() # 3/19
# fwrite(apoecs_clin,fp(out,'res_lm_apoecs_adpathology_apoe4_2_correction.csv.gz'))

# Original results:
#     cs_id             outcome        padj                                                                                                              gwas_source
#    <char>              <char>       <num>                                                                                                                   <char>
# 1: set_27             caa_4gp 0.004708944 AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021
# 2: set_27            caa_neo4 0.007830511 AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021
# 3: set_27             tdp_st4 0.042915291 AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021
# 4: set_31 age_first_ad_dx_num 0.020960086                                                                                                            AD_Bellenguez
# 5: set_57        tangles_sqrt 0.003434581                           AD_Bellenguez; AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021
#                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     molecular_outcome
#                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                <char>
# 1: AC_DeJager_eQTL_ENSG00000224916; AC_DeJager_eQTL_ENSG00000234906; BM_10_MSBB_eQTL_ENSG00000104853; Metabrain_Basalganglia_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cerebellum_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000104853; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000234906; Mic_DeJager_eQTL_ENSG00000130208; Mic_DeJager_eQTL_ENSG00000130203; Mic_mega_eQTL_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000130208; STARNET_eQTL_Mac_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000267467; STARNET_eQTL_Mac_ENSG00000224916; STARNET_eQTL_Mac_ENSG00000234906; DeJager_Mic_ENSG00000130203; Kellis_Mic_ENSG00000130203; DeJager_Mic_ENSG00000130208; Kellis_Mic_ENSG00000130208
# 2: AC_DeJager_eQTL_ENSG00000224916; AC_DeJager_eQTL_ENSG00000234906; BM_10_MSBB_eQTL_ENSG00000104853; Metabrain_Basalganglia_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cerebellum_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000104853; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000234906; Mic_DeJager_eQTL_ENSG00000130208; Mic_DeJager_eQTL_ENSG00000130203; Mic_mega_eQTL_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000130208; STARNET_eQTL_Mac_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000267467; STARNET_eQTL_Mac_ENSG00000224916; STARNET_eQTL_Mac_ENSG00000234906; DeJager_Mic_ENSG00000130203; Kellis_Mic_ENSG00000130203; DeJager_Mic_ENSG00000130208; Kellis_Mic_ENSG00000130208
# 3: AC_DeJager_eQTL_ENSG00000224916; AC_DeJager_eQTL_ENSG00000234906; BM_10_MSBB_eQTL_ENSG00000104853; Metabrain_Basalganglia_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cerebellum_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000104853; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000234906; Mic_DeJager_eQTL_ENSG00000130208; Mic_DeJager_eQTL_ENSG00000130203; Mic_mega_eQTL_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000130208; STARNET_eQTL_Mac_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000267467; STARNET_eQTL_Mac_ENSG00000224916; STARNET_eQTL_Mac_ENSG00000234906; DeJager_Mic_ENSG00000130203; Kellis_Mic_ENSG00000130203; DeJager_Mic_ENSG00000130208; Kellis_Mic_ENSG00000130208
# 4:                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               AC_DeJager_eQTL_ENSG00000189114; DLPFC_DeJager_eQTL_ENSG00000189114; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000189114; PCC_DeJager_eQTL_ENSG00000189114; ROSMAP_AC_ENSG00000189114; ROSMAP_DLPFC_ENSG00000189114; ROSMAP_PCC_ENSG00000189114
# 5:                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 Ast_10_Kellis_eQTL_ENSG00000160007

[1] "set_57" "set_27" "set_31"

In [52]:
# Save the significantly associated sets
apoecs_clin_sig <- apoecs_clin[padj<0.05][!outcome %in% c('apoe4_dose','apoe2_dose')]
write.table(apoecs_clin_sig, file = '/mnt/lustre/home/yl4437/xqtl_flagship/APOE/res_lm_apoecs_adpathology_apoe4_2_adj_sig.txt',
            sep = '\t', quote = FALSE, row.names = FALSE, col.names = TRUE)

In [24]:
# Set27 is the cerebral amyloid angiopathy (CAA) set of Fujita et al.?
# apoecs_clin[pos(variant_id2) == 44946027] # Yes

### Regression 2: Adjusting for APOE4 only
Repeat the same regression framework but adjusting only for APOE4 dosage (not APOE2) to evaluate sensitivity of results to APOE2 correction

In [40]:
# Adjusting for APOE4 only
apoecs_clin2 <- rbindlist(lapply(covs_lin, function(outc){
  message(outc)
  if(outc %in% atlastvisit){
    donors_apoecs[,summary(lm(unlist(.SD)~dose+msex+educ+age_at_visit_num+apoe4_dose))$coefficients|>data.table(keep.rownames = 'cov'),
                  by='variant_id2',.SDcols=outc][cov=='dose'][,outcome:=outc]
  }else if(outc %in% beforedeath){
    donors_apoecs[,summary(lm(unlist(.SD)~dose+msex+educ+apoe4_dose))$coefficients|>data.table(keep.rownames = 'cov'),
                  by='variant_id2',.SDcols=outc][cov=='dose'][,outcome:=outc]
  }else{
    donors_apoecs[,summary(lm(unlist(.SD)~dose+msex+educ+age_death+apoe4_dose))$coefficients|>data.table(keep.rownames = 'cov'),
                  by='variant_id2',.SDcols=outc][cov=='dose'][,outcome:=outc]
  }
}))

apoecs_clin2[,zscore:=`t value`]
apoecs_clin2[,pval:=`Pr(>|t|)`]

ceradsc_pos



braaksc06

dcfdx_lv_no6

cts_mmse30_lv

age_death

cogdx_mergenum

apoe4_dose

Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: summary may be unreliable”
Warning message in summary.lm(lm(unlist(.SD) ~ dose + msex + educ + age_death + :
“essentially perfect fit: sum

In [41]:
apoecs_clogit2 <- rbindlist(lapply(covs_logis, function(outc){
  message(outc)
  donors_apoecs[,summary(glm(unlist(.SD)~dose+msex+educ+age_death+apoe4_dose,family = binomial))$coefficients|>data.table(keep.rownames = 'cov'),
                by='variant_id2',.SDcols=outc][cov=='dose'][,outcome:=outc]
}))

apoecs_clogit2[,zscore:=`z value`]
apoecs_clogit2[,pval:=`Pr(>|z|)`]

MCIvsNL



ADvsNL



In [42]:
apoecs_clin2 <- rbind(apoecs_clin2, apoecs_clogit2, fill = TRUE)
apoecs_clin2 <- merge(apoecs_clin2, indep_cs, by = 'variant_id2')
apoecs_clin2[,padj:=p.adjust(pval,n=length(unique(indep_cs$cs_name))),by=c('outcome','variant_id2')]
head(apoecs_clin2[padj<0.05])

variant_id2,cov,Estimate,Std. Error,t value,Pr(>|t|),outcome,zscore,pval,z value,Pr(>|z|),cs_name,variant_id,gwas_source,molecular_outcome,cs_id,chr,pos,padj
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
chr19:44723473_T_C,dose,-0.1501612,0.04979175,-3.015784,0.002621997,apoe2_dose,-3.015784,0.002621997,NA,NA,ind_set_after_between_purity_267,chr19:44723473:T:C,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44723473,0.04981794
chr19:44724393_G_T,dose,-0.1501612,0.04979175,-3.015784,0.002621997,apoe2_dose,-3.015784,0.002621997,NA,NA,ind_set_after_between_purity_267,chr19:44724393:G:T,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44724393,0.04981794
chr19:44724456_A_G,dose,-0.1501612,0.04979175,-3.015784,0.002621997,apoe2_dose,-3.015784,0.002621997,NA,NA,ind_set_after_between_purity_267,chr19:44724456:A:G,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44724456,0.04981794
chr19:44724557_C_T,dose,-0.1501612,0.04979175,-3.015784,0.002621997,apoe2_dose,-3.015784,0.002621997,NA,NA,ind_set_after_between_purity_267,chr19:44724557:C:T,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44724557,0.04981794
chr19:44724638_C_A,dose,-0.1501612,0.04979175,-3.015784,0.002621997,apoe2_dose,-3.015784,0.002621997,NA,NA,ind_set_after_between_purity_267,chr19:44724638:C:A,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44724638,0.04981794
chr19:44724664_G_A,dose,-0.1501612,0.04979175,-3.015784,0.002621997,apoe2_dose,-3.015784,0.002621997,NA,NA,ind_set_after_between_purity_267,chr19:44724664:G:A,AD_Wightman_Full_2021,monocyte_ROSMAP_eQTL_ENSG00000105321,set_267,chr19,44724664,0.04981794


In [ ]:
# Filter significant results and inspect which CS and outcomes are significant when only adjusting for APOE4
# apoecs_clin2[!outcome %in% c('apoe4_dose','apoe2_dose')][padj<0.05]
apoecs_clin2[padj<0.05][!outcome%in%c('apoe4_dose','apoe2_dose')]$cs_id|>unique() # 4/19 
# fwrite(apoecs_clin2,fp(out,'res_lm_apoecs_adpathology_apoe4_correction.csv.gz'))

[1] "set_57" "set_26" "set_27" "set_31"

In [44]:
unique(apoecs_clin2[padj<0.05][!outcome%in%c('apoe4_dose','apoe2_dose')][order(cs_id,cov,padj)][,.(cs_id,outcome,variant_id2,padj,gwas_source,molecular_outcome)],by=c('cs_id','outcome'))

# Original results:
#     cs_id             outcome              variant_id2        padj
#    <char>              <char>                   <char>       <num>
# 1: set_26 age_first_ad_dx_num       chr19:44918487_G_T 0.046608400
# 2: set_27             caa_4gp       chr19:44954310_T_C 0.002833505
# 3: set_27            caa_neo4       chr19:44954310_T_C 0.004877779
# 4: set_27             tdp_st4 chr19:44999110_TAAAA_TAA 0.042966548
# 5: set_31 age_first_ad_dx_num   chr19:45114770_ACCC_AC 0.020474889
# 6: set_57        tangles_sqrt       chr19:44840322_G_A 0.020962988

cs_id,outcome,variant_id2,padj,gwas_source,molecular_outcome
<chr>,<chr>,<chr>,<dbl>,<chr>,<chr>
set_26,age_first_ad_dx_num,chr19:44918487_G_T,0.041552422,AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021,AC_DeJager_eQTL_ENSG00000130208; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000130208; STARNET_eQTL_Mac_ENSG00000130208; ROSMAP_AC_ENSG00000130208; ROSMAP_DLPFC_ENSG00000130208
set_27,caa_4gp,chr19:44954310_T_C,0.003646053,AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021,AC_DeJager_eQTL_ENSG00000224916; AC_DeJager_eQTL_ENSG00000234906; BM_10_MSBB_eQTL_ENSG00000104853; Metabrain_Basalganglia_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cerebellum_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000104853; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000234906; Mic_DeJager_eQTL_ENSG00000130208; Mic_DeJager_eQTL_ENSG00000130203; Mic_mega_eQTL_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000130208; STARNET_eQTL_Mac_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000267467; STARNET_eQTL_Mac_ENSG00000224916; STARNET_eQTL_Mac_ENSG00000234906; DeJager_Mic_ENSG00000130203; Kellis_Mic_ENSG00000130203; DeJager_Mic_ENSG00000130208; Kellis_Mic_ENSG00000130208
set_27,caa_neo4,chr19:44954310_T_C,0.007036152,AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021,AC_DeJager_eQTL_ENSG00000224916; AC_DeJager_eQTL_ENSG00000234906; BM_10_MSBB_eQTL_ENSG00000104853; Metabrain_Basalganglia_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cerebellum_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000104853; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000234906; Mic_DeJager_eQTL_ENSG00000130208; Mic_DeJager_eQTL_ENSG00000130203; Mic_mega_eQTL_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000130208; STARNET_eQTL_Mac_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000267467; STARNET_eQTL_Mac_ENSG00000224916; STARNET_eQTL_Mac_ENSG00000234906; DeJager_Mic_ENSG00000130203; Kellis_Mic_ENSG00000130203; DeJager_Mic_ENSG00000130208; Kellis_Mic_ENSG00000130208
set_27,tdp_st4,chr19:44999110_TAAAA_TAA,0.039304996,AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021; AD_Wightman_ExcludingUKBand23andME_2021,AC_DeJager_eQTL_ENSG00000224916; AC_DeJager_eQTL_ENSG00000234906; BM_10_MSBB_eQTL_ENSG00000104853; Metabrain_Basalganglia_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cerebellum_chr19_41840000_47960000_ENSG00000234906; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000104853; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000234906; Mic_DeJager_eQTL_ENSG00000130208; Mic_DeJager_eQTL_ENSG00000130203; Mic_mega_eQTL_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000130208; STARNET_eQTL_Mac_ENSG00000130203; STARNET_eQTL_Mac_ENSG00000267467; STARNET_eQTL_Mac_ENSG00000224916; STARNET_eQTL_Mac_ENSG00000234906; DeJager_Mic_ENSG00000130203; Kellis_Mic_ENSG00000130203; DeJager_Mic_ENSG00000130208; Kellis_Mic_ENSG00000130208
set_31,age_first_ad_dx_num,chr19:45114770_ACCC_AC,0.021032084,AD_Bellenguez,AC_DeJager_eQTL_ENSG00000189114; DLPFC_DeJager_eQTL_ENSG00000189114; Metabrain_Cortex_chr19_41840000_47960000_ENSG00000189114; PCC_DeJager_eQTL_ENSG00000189114; ROSMAP_AC_ENSG00000189114; ROSMAP_DLPFC_ENSG00000189114; ROSMAP_PCC_ENSG00000189114
set_57,tangles_sqrt,chr19:44840322_G_A,0.020784285,AD_Bellenguez; AD_Kunkle_Stage1_2019; AD_Wightman_Full_2021; AD_Wightman_Excluding23andMe_2021,Ast_10_Kellis_eQTL_ENSG00000160007


In [53]:
# Save the significantly associated sets
apoecs_clin2_sig <- apoecs_clin2[padj<0.05][!outcome %in% c('apoe4_dose','apoe2_dose')]
write.table(apoecs_clin2_sig, file = '/mnt/lustre/home/yl4437/xqtl_flagship/APOE/res_lm_apoecs_adpathology_apoe4_adj_sig.txt',
            sep = '\t', quote = FALSE, row.names = FALSE, col.names = TRUE)

Follow-up: Is set_26 driven by APOE2?

In [ ]:
# Set26 is APOE2?
apoecs_clin2[cs_id=='set_26'&pos(variant_id2)==44908822] # No

donors_apoecs[variant_id2=='chr19:44918487_G_T']$apoe2_r[1] # -0.22
donors_apoecs[variant_id2=='chr19:44918487_G_T']$apoe4_r[1] # -0.24

variant_id2,cov,Estimate,Std. Error,t value,Pr(>|t|),outcome,zscore,pval,z value,Pr(>|z|),cs_name,variant_id,gwas_source,molecular_outcome,cs_id,chr,pos,padj
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>


[1] -0.2151048

[1] -0.2410754